## Import

In [ ]:
import pandas as pd
import random
import os
import numpy as np

from matplotlib import pyplot as plt

import shap

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import f1_score

In [ ]:
class CFG:
    SEED = 42

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(CFG.SEED) # Seed 고정

## Data Load

In [ ]:
X_train = pd.read_csv('data/X_train_fine.csv')
X_test = pd.read_csv('data/X_test_fine.csv')
y_train = pd.read_csv('data/y_train_fine.csv')
submit = pd.read_csv('data/sample_submission.csv')
y_train = y_train['class']

In [ ]:
X_sex = pd.concat([X_train,X_test.loc[[103]]]).reset_index(drop=True)

In [ ]:
X_sex_list = X_sex.transpose().corr().sort_values(by=342, ascending=False).iloc[:300,-1:].index.to_list()
X_sex_list.remove(342)

In [ ]:
X_train_01 = X_train.loc[X_sex_list,:]
y_train_01 = pd.DataFrame(y_train).loc[X_sex_list,:]['class']

In [ ]:
def evaluate_macroF1_lgb(truth, predictions):  
    pred_labels = predictions.reshape(len(np.unique(truth)),-1).argmax(axis=0)
    f1 = f1_score(truth, pred_labels, average='macro')
    return ('macroF1', f1, True) 

In [ ]:
import shap

# DF, based on which importance is checked
X_importance = X_test

# Explain model predictions using shap library:
model = CatBoostClassifier(random_state=42,eval_metric="TotalF1:average=Macro").fit(X_train_01, y_train_01)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_importance)

# Plot summary_plot as barplot:
shap.summary_plot(shap_values, X_importance, plot_type='bar')

shap_sum = np.abs(shap_values).mean(axis=1)[1,:]
importance_df = pd.DataFrame([X_importance.columns.tolist(), shap_sum.tolist()]).T
importance_df.columns = ['column_name', 'shap_importance']
importance_df = importance_df.sort_values('shap_importance', ascending=False)
importance_df

In [ ]:
features_selected = importance_df.query('shap_importance < 0.000001').column_name.tolist()
X_train = X_train[features_selected]
X_test = X_test[features_selected]
X_train.shape, X_test.shape

### Model_Baseline

In [ ]:
def evaluate_macroF1_lgb(truth, predictions):  
    pred_labels = predictions.reshape(len(np.unique(truth)),-1).argmax(axis=0)
    f1 = f1_score(truth, pred_labels, average='macro')
    return ('macroF1', f1, True) 

In [ ]:
model = CatBoostClassifier(random_state=42,eval_metric="TotalF1:average=Macro")

In [ ]:
model.fit(X_train, y_train)

In [ ]:
submit['class'] = model.predict(X_test)

In [ ]:
submit.loc[[103]]

In [ ]:
Best_sub = pd.read_csv('submissions/baseline12.csv')

## Emsemble

In [ ]:
# Last_submmit.to_csv('submissions/Fine_01.csv', index=False)